In [1]:
"""
Distributed System Simulation
Lamport Logical Clocks + Chandy-Lamport Global Snapshot

Run:
    python distributed_sim.py

Python version: 3.9+
Only standard-library modules are used.
"""

from __future__ import annotations

import queue
import random
import threading
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple


PROCESS_IDS = ("P0", "P1", "P2")
INITIAL_BALANCE = 100
EXPECTED_TOTAL = INITIAL_BALANCE * len(PROCESS_IDS)


@dataclass
class Message:
    kind: str
    source: str
    destination: str
    clock: int
    amount: int = 0


class Process:
    """One simulated distributed process, executed by its own thread."""

    def __init__(self, pid: str, network: Dict[str, "Process"], log_lock: threading.Lock):
        self.pid = pid
        self.network = network
        self.log_lock = log_lock
        self.inbox: queue.Queue[Message] = queue.Queue()

        self.balance = INITIAL_BALANCE
        self.lamport_clock = 0

        self.snapshot_taken = False
        self.recorded_state: Optional[dict] = None
        self.recording: Dict[str, bool] = {p: False for p in PROCESS_IDS if p != pid}
        self.channel_states: Dict[str, List[dict]] = {
            p: [] for p in PROCESS_IDS if p != pid
        }

        self.actions_done = threading.Event()
        self.stop_event = threading.Event()
        self.thread: Optional[threading.Thread] = None

    def log(self, message: str) -> None:
        with self.log_lock:
            print(f"[{self.pid} | LC={self.lamport_clock:02d}] {message}")

    def tick(self) -> int:
        self.lamport_clock += 1
        return self.lamport_clock

    def on_receive_clock(self, received_clock: int) -> int:
        self.lamport_clock = max(self.lamport_clock, received_clock) + 1
        return self.lamport_clock

    def send_transfer(self, destination: str, amount: int) -> None:
        if amount <= 0:
            raise ValueError("Transfer amount must be positive.")
        if self.balance < amount:
            raise ValueError(f"{self.pid} does not have enough balance to transfer {amount}.")

        clock = self.tick()
        self.balance -= amount
        message = Message("TRANSFER", self.pid, destination, clock, amount)
        self.network[destination].inbox.put(message)
        self.log(f"SEND {amount} units -> {destination}; balance={self.balance}")

    def initiate_snapshot(self) -> None:
        if self.snapshot_taken:
            return

        self.tick()
        self.snapshot_taken = True
        self.recorded_state = {"balance": self.balance, "lamport_clock": self.lamport_clock}

        # The initiator records every incoming channel from this point onward.
        for source in self.recording:
            self.recording[source] = True
            self.channel_states[source] = []

        self.log(f"SNAPSHOT initiated; recorded balance={self.balance}")
        for destination in PROCESS_IDS:
            if destination != self.pid:
                self.send_marker(destination)

    def send_marker(self, destination: str) -> None:
        clock = self.tick()
        marker = Message("MARKER", self.pid, destination, clock)
        self.network[destination].inbox.put(marker)
        self.log(f"SEND-MARKER -> {destination}")

    def handle_marker(self, message: Message) -> None:
        source = message.source

        if not self.snapshot_taken:
            # First marker: record local state; the channel carrying this marker is empty.
            self.tick()
            self.snapshot_taken = True
            self.recorded_state = {
                "balance": self.balance,
                "lamport_clock": self.lamport_clock,
            }
            self.channel_states[source] = []
            self.recording[source] = False

            # Start recording all other incoming channels.
            for other_source in self.recording:
                if other_source != source:
                    self.recording[other_source] = True
                    self.channel_states[other_source] = []

            self.log(
                f"SNAPSHOT recorded local balance={self.balance}; "
                f"channel {source}->{self.pid}=EMPTY"
            )

            # Forward markers on every outgoing channel.
            for destination in PROCESS_IDS:
                if destination != self.pid:
                    self.send_marker(destination)
        else:
            # A later marker closes the corresponding incoming channel.
            self.recording[source] = False
            self.log(f"MARKER received from {source}; channel recording closed")

    def handle_message(self, message: Message) -> None:
        self.on_receive_clock(message.clock)

        if message.kind == "MARKER":
            self.handle_marker(message)
            return

        if message.kind == "TRANSFER":
            # Record the message if this channel was still open in our snapshot.
            if self.snapshot_taken and self.recording.get(message.source, False):
                self.channel_states[message.source].append(
                    {"amount": message.amount, "from": message.source, "to": self.pid}
                )

            self.balance += message.amount
            self.log(
                f"RECEIVE {message.amount} units <- {message.source}; "
                f"balance={self.balance}"
            )
            return

        raise ValueError(f"Unknown message type: {message.kind}")

    def drain_one_message(self) -> bool:
        try:
            message = self.inbox.get_nowait()
        except queue.Empty:
            return False
        self.handle_message(message)
        self.inbox.task_done()
        return True

    def run(self, actions: List[Tuple[str, int, str]], snapshot_after: Optional[int],
            delay_rng: random.Random, max_delay: float) -> None:
        """
        actions: list of (destination, amount, description)
        snapshot_after: take snapshot after this many local transfer actions;
                        None means this process does not initiate it.
        """
        for index, (destination, amount, description) in enumerate(actions, start=1):
            # Process incoming traffic while waiting, so channels remain asynchronous.
            wait_time = delay_rng.uniform(0.01, max_delay)
            deadline = time.monotonic() + wait_time
            while time.monotonic() < deadline and not self.stop_event.is_set():
                if not self.drain_one_message():
                    time.sleep(0.002)

            self.log(f"LOCAL computation: {description}")
            self.send_transfer(destination, amount)

            if snapshot_after == index:
                self.initiate_snapshot()

            # Give incoming messages a chance to be handled between local events.
            for _ in range(3):
                if not self.drain_one_message():
                    break

        self.actions_done.set()

        # Continue handling incoming messages and markers until coordinator stops us.
        while not self.stop_event.is_set():
            if not self.drain_one_message():
                time.sleep(0.002)


class DistributedSimulation:
    def __init__(self, seed: int, initiator: str, max_delay: float = 0.06):
        if initiator not in PROCESS_IDS:
            raise ValueError(f"Initiator must be one of {PROCESS_IDS}")

        self.seed = seed
        self.initiator = initiator
        self.max_delay = max_delay
        self.log_lock = threading.Lock()
        self.processes: Dict[str, Process] = {}

        # Create all process objects first, then connect their network references.
        for pid in PROCESS_IDS:
            self.processes[pid] = Process(pid, self.processes, self.log_lock)

        self.actions = {
            "P0": [("P1", 10, "prepare transfer to P1"),
                   ("P2", 20, "send another transfer to P2")],
            "P1": [("P2", 15, "prepare transfer to P2"),
                   ("P0", 5, "send another transfer to P0")],
            "P2": [("P0", 5, "prepare transfer to P0"),
                   ("P1", 10, "send another transfer to P1")],
        }

    def snapshot_is_complete(self) -> bool:
        for process in self.processes.values():
            if not process.snapshot_taken:
                return False
            if any(process.recording.values()):
                return False
        return True

    def all_queues_empty(self) -> bool:
        return all(process.inbox.empty() for process in self.processes.values())

    def run(self) -> bool:
        print("\n" + "=" * 72)
        print(
            f"RUN: seed={self.seed}, snapshot initiator={self.initiator}, "
            f"initial total={EXPECTED_TOTAL}"
        )
        print("=" * 72)

        for pid, process in self.processes.items():
            rng = random.Random(self.seed + PROCESS_IDS.index(pid) * 1000)
            snapshot_after = 1 if pid == self.initiator else None
            process.thread = threading.Thread(
                target=process.run,
                args=(self.actions[pid], snapshot_after, rng, self.max_delay),
                name=f"Thread-{pid}",
                daemon=True,
            )
            process.thread.start()

        # Wait until all local work and all marker/channel recording are complete.
        deadline = time.monotonic() + 10.0
        while time.monotonic() < deadline:
            all_actions_done = all(p.actions_done.is_set() for p in self.processes.values())
            if all_actions_done and self.snapshot_is_complete() and self.all_queues_empty():
                # Let worker threads finish any message they have already dequeued.
                time.sleep(0.03)
                if self.all_queues_empty():
                    break
            time.sleep(0.005)
        else:
            print("ERROR: simulation timed out before snapshot completion.")
            self.stop_all()
            return False

        self.stop_all()
        for process in self.processes.values():
            if process.thread:
                process.thread.join(timeout=1.0)

        return self.verify_snapshot()

    def stop_all(self) -> None:
        for process in self.processes.values():
            process.stop_event.set()

    def verify_snapshot(self) -> bool:
        print("\n--- GLOBAL SNAPSHOT ---")
        local_total = 0
        in_transit_total = 0

        for pid in PROCESS_IDS:
            process = self.processes[pid]
            state = process.recorded_state
            if state is None:
                print(f"ERROR: {pid} did not record a local state.")
                return False
            balance = state["balance"]
            local_total += balance
            print(f"Local state {pid}: balance={balance}, LC={state['lamport_clock']}")

        for destination in PROCESS_IDS:
            process = self.processes[destination]
            for source, messages in process.channel_states.items():
                channel_amount = sum(item["amount"] for item in messages)
                in_transit_total += channel_amount
                print(
                    f"Channel {source} -> {destination}: "
                    f"{messages if messages else 'EMPTY'}"
                )

        recorded_total = local_total + in_transit_total
        print("\n--- CONSISTENCY CHECK ---")
        print(f"Sum of recorded local balances : {local_total}")
        print(f"Sum of in-transit messages     : {in_transit_total}")
        print(f"Recorded global total          : {recorded_total}")
        print(f"Expected total                 : {EXPECTED_TOTAL}")

        passed = recorded_total == EXPECTED_TOTAL
        print(f"RESULT: {'PASS' if passed else 'FAIL'}")
        return passed


def main() -> None:
    test_cases = [
        (1, "P0"),
        (7, "P1"),
        (42, "P2"),
        (99, "P0"),
        (123, "P1"),
    ]

    results = []
    for seed, initiator in test_cases:
        simulation = DistributedSimulation(seed=seed, initiator=initiator)
        results.append((seed, initiator, simulation.run()))

    print("\n" + "=" * 72)
    print("TEST SUMMARY")
    print("=" * 72)
    for seed, initiator, passed in results:
        print(
            f"Seed={seed:<3} Initiator={initiator:<2} "
            f"{'PASS' if passed else 'FAIL'}"
        )

    passed_count = sum(1 for _, _, passed in results if passed)
    print(f"\n{passed_count}/{len(results)} test scenarios passed.")
    if passed_count != len(results):
        raise SystemExit(1)


if __name__ == "__main__":
    main()



RUN: seed=1, snapshot initiator=P0, initial total=300
[P0 | LC=00] LOCAL computation: prepare transfer to P1
[P0 | LC=01] SEND 10 units -> P1; balance=90
[P0 | LC=02] SNAPSHOT initiated; recorded balance=90
[P0 | LC=03] SEND-MARKER -> P1
[P0 | LC=04] SEND-MARKER -> P2
[P1 | LC=02] RECEIVE 10 units <- P0; balance=110
[P1 | LC=05] SNAPSHOT recorded local balance=110; channel P0->P1=EMPTY
[P1 | LC=06] SEND-MARKER -> P0
[P1 | LC=07] SEND-MARKER -> P2
[P2 | LC=06] SNAPSHOT recorded local balance=100; channel P0->P2=EMPTY
[P2 | LC=07] SEND-MARKER -> P0
[P2 | LC=08] SEND-MARKER -> P1
[P2 | LC=09] MARKER received from P1; channel recording closed
[P1 | LC=09] MARKER received from P2; channel recording closed
[P0 | LC=07] MARKER received from P1; channel recording closed
[P0 | LC=08] MARKER received from P2; channel recording closed
[P2 | LC=09] LOCAL computation: prepare transfer to P0
[P2 | LC=10] SEND 5 units -> P0; balance=95
[P0 | LC=11] RECEIVE 5 units <- P2; balance=95
[P1 | LC=09] LOCA